In [1]:
import requests
import json
import os
import time
import yaml
from dotenv import load_dotenv
import string
load_dotenv()

True

In [2]:
def poll_results(task_id):
    """Poll the API for task results and return the data"""
    
    print("Polling for task results...")
    max_attempts = 100
    attempts = 0
    
    while attempts < max_attempts:
        attempts += 1
        response = requests.get(f"{API_BASE_URL}/tasks/{task_id}")
        
        if response.status_code == 200:
            data = response.json()['data']
            status = data.get("status")
            
            print(f"Task status: {status}")
            
            if status == "completed":
                print("Task completed!")
                # print("Results:")
                # print(json.dumps(data.get("results"), indent=2))
                
                # Check for simulator interactions
                # simulator_interactions = data.get("simulator_interactions", [])
                # if simulator_interactions:
                #     print("\nUser Simulator Interactions:")
                #     print(json.dumps(simulator_interactions, indent=2))
                return data
            elif status == "failed":
                print("Task failed!")
                print("Error:")
                print(data.get("error"))
                return data
            elif status == "cancelled":
                print("Task was cancelled")
                return data
        
        # Wait before polling again
        time.sleep(5)
    
    print("Max polling attempts reached. Task may still be running.")
    return None

In [3]:
def get_template_placeholders(template):

    placeholders = template.get('placeholders', {})
    
    # Return placeholder info in order
    placeholder_list = []
    for placeholder_name, placeholder_config in placeholders.items():
        placeholder_list.append({
            'name': placeholder_name,
            'config': placeholder_config
        })
    
    return placeholder_list

In [4]:
template = """
simple_test:
    metadata:
      name: "Simple Test"
      description: "This is a simple test for Google search"
      tags: ["test", "simple", "google"]
    
    placeholders:
      query:
        type: string
        description: "The search query"
        required: true
        ui_type: text
        emoji: "🔍"
    
    system_placeholders:
      - session_id
    
    config:
      laminar_api_key: ""
      laminar_base_url: ""
      laminar_http_port: 0
      laminar_grpc_port: 0
      simulator_provider: "google"
      simulator_model: "gemini-2.0-flash"
      simulator_temperature: 0.5
      custom_actions: []
      simulator_task: ""
      browser_config:
        keep_alive: true
        headless: true
        viewport:
          width: 1920
          height: 1080
    
    tasks:
      - name: "Simple test"
        prompt: "Go to google and search for ${query}"
        max_steps: 15
        output_model_fields: null
        exclude_actions: []
        llm_provider: "google"
        llm_model: "gemini-2.0-flash"
        llm_temperature: 0.1
        enable_memory: true
        memory_interval: 5
        initial_actions: []
        
        report_config:
          provider: "google"
          model: "gemini-2.0-flash"
          temperature: 0.2
          is_report_reasoning: false
          use_vision_for_report: false
          report_folder: ""
          extend_report_system_message: ""
"""

In [5]:
data = yaml.safe_load(template)["simple_test"]
placeholders = get_template_placeholders(data)

In [6]:
data

{'metadata': {'name': 'Simple Test',
  'description': 'This is a simple test for Google search',
  'tags': ['test', 'simple', 'google']},
 'placeholders': {'query': {'type': 'string',
   'description': 'The search query',
   'required': True,
   'ui_type': 'text',
   'emoji': '🔍'}},
 'system_placeholders': ['session_id'],
 'config': {'laminar_api_key': '',
  'laminar_base_url': '',
  'laminar_http_port': 0,
  'laminar_grpc_port': 0,
  'simulator_provider': 'google',
  'simulator_model': 'gemini-2.0-flash',
  'simulator_temperature': 0.5,
  'custom_actions': [],
  'simulator_task': '',
  'browser_config': {'keep_alive': True,
   'headless': True,
   'viewport': {'width': 1920, 'height': 1080}}},
 'tasks': [{'name': 'Simple test',
   'prompt': 'Go to google and search for ${query}',
   'max_steps': 15,
   'output_model_fields': None,
   'exclude_actions': [],
   'llm_provider': 'google',
   'llm_model': 'gemini-2.0-flash',
   'llm_temperature': 0.1,
   'enable_memory': True,
   'memory_i

In [7]:
query = "Sơn Tùng MTP"
placeholder_dict = {}
for i, placeholder_info in enumerate(placeholders):
    print(i, placeholder_info)
    placeholder_name = placeholder_info['name']
    placeholder_dict[placeholder_name] = query

0 {'name': 'query', 'config': {'type': 'string', 'description': 'The search query', 'required': True, 'ui_type': 'text', 'emoji': '🔍'}}


In [8]:
def substitute_template_variables(obj, substitutions):
    """Recursively substitute ${variable} patterns in the template"""
    if isinstance(obj, dict):
        result = {}
        for key, value in obj.items():
            result[key] = substitute_template_variables(value, substitutions)
        return result
    elif isinstance(obj, list):
        return [substitute_template_variables(item, substitutions) for item in obj]
    elif isinstance(obj, str):
        # Use string.Template for safe substitution
        template = string.Template(obj)
        try:
            return template.substitute(substitutions)
        except KeyError as e:
            # If placeholder is missing, keep the original
            print(f"Warning: Missing placeholder {e} in template")
            return obj
    else:
        return obj

In [9]:
def create_payload_from_template(template, placeholder_values, case_id):
    
    # Get root report folder from environment
    root_report = os.getenv("ROOT_REPORT", "E:/official_DopikAI/ai-agent-tester/reports")
    report_folder_path = f"{root_report}/{case_id}"
    
    # Prepare substitutions with user values and system-generated values
    substitutions = dict(placeholder_values)
    substitutions.update({
        'session_id': f"test-session-id-{case_id}",
        'report_folder': report_folder_path
    })
    
    # Convert YAML template to the expected API format
    payload = {
        'tasks': template.get('tasks', []),
        'custom_actions': template.get("config", {}).get("custom_actions", []),
        'simulator_task': template.get("config", {}).get("simulator_task", ""),
        'browser_config': template.get('config', {}).get('browser_config', {})
    }
    
    # Add config values
    config = template.get('config', {})
    payload.update({
        'laminar_api_key': os.getenv("LAMINAR_API_KEY", config.get('laminar_api_key', "")),
        'laminar_base_url': os.getenv("LAMINAR_BASE_URL", config.get('laminar_base_url', "")),
        'laminar_http_port': int(os.getenv("LAMINAR_HTTP_PORT", config.get('laminar_http_port', 0)) or 0),
        'laminar_grpc_port': int(os.getenv("LAMINAR_GRPC_PORT", config.get('laminar_grpc_port', 0)) or 0),
        'simulator_provider': config.get('simulator_provider', 'google'),
        'simulator_model': config.get('simulator_model', 'gemini-2.0-flash'),
        'simulator_temperature': config.get('simulator_temperature', 0.0),
        'session_id': substitutions['session_id']
    })
    
    # Apply substitutions using string Template (safer than format)
    payload = substitute_template_variables(payload, substitutions)
    
    return payload

In [10]:
payload = create_payload_from_template(data, placeholder_dict, "new_api")

In [11]:
payload

{'tasks': [{'name': 'Simple test',
   'prompt': 'Go to google and search for Sơn Tùng MTP',
   'max_steps': 15,
   'output_model_fields': None,
   'exclude_actions': [],
   'llm_provider': 'google',
   'llm_model': 'gemini-2.0-flash',
   'llm_temperature': 0.1,
   'enable_memory': True,
   'memory_interval': 5,
   'initial_actions': [],
   'report_config': {'provider': 'google',
    'model': 'gemini-2.0-flash',
    'temperature': 0.2,
    'is_report_reasoning': False,
    'use_vision_for_report': False,
    'report_folder': '',
    'extend_report_system_message': ''}}],
 'custom_actions': [],
 'simulator_task': '',
 'browser_config': {'keep_alive': True,
  'headless': True,
  'viewport': {'width': 1920, 'height': 1080}},
 'laminar_api_key': '',
 'laminar_base_url': '',
 'laminar_http_port': 0,
 'laminar_grpc_port': 0,
 'simulator_provider': 'google',
 'simulator_model': 'gemini-2.0-flash',
 'simulator_temperature': 0.5,
 'session_id': 'test-session-id-new_api'}

In [17]:
API_BASE_URL = "http://localhost:8081"
response = requests.post(f"{API_BASE_URL}/tasks/run", json=payload)
# Print response
print(f"Status code: {response.status_code}")
print(f"Response: {response.json()}")
if response.status_code == 200:
    # Extract task ID
    task_id = response.json()['data']["message"].split(": ")[1]
    print(f"Task ID: {task_id}")
    
    # Poll for results
    poll_results(task_id)
else:
    print(f"Failed to start task: {response.text}")

Status code: 200
Response: {'data': {'message': 'Task started with ID: 37f60f3a-f8d4-4463-b879-dfbcff28d1a6'}, 'message': 'Success'}
Task ID: 37f60f3a-f8d4-4463-b879-dfbcff28d1a6
Polling for task results...
Task status: completed
Task completed!
